# Data Engineering Task 2: SQL-Based Sales Analysis
**Author:** Parth Gehlot  
**Objective:** Analyze Superstore sales data using SQLite and Python to perform filtering, aggregation, and business intelligence queries.

---

## Step 1 & 2: Database Initialization and Schema Exploration
**Process:** We begin by reading the raw `Sample - Superstore.csv` file using Pandas, standardizing the column headers for SQL compatibility, and ingesting the dataframe into a local SQLite database table named `sales`.

In [1]:
import pandas as pd
import sqlite3

# Load the CSV
df = pd.read_csv('Sample - Superstore.csv', encoding='windows-1252') # Superstore often needs this encoding

# Clean up column names (replace spaces with underscores for easier SQL querying)
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

# Create a connection to a local SQLite database
conn = sqlite3.connect('superstore_sales.db')

# Write the dataframe to a SQL table named 'sales'
df.to_sql('sales', conn, if_exists='replace', index=False)
print("Data loaded successfully!")

Data loaded successfully!


In [2]:
# View sample data
query_explore = """
SELECT Row_ID, Order_ID, Order_Date, Customer_Name, Region, Category, Sales 
FROM sales 
LIMIT 5;
"""
display(pd.read_sql_query(query_explore, conn))

,Row_ID,Order_ID,Order_Date,Customer_Name,Region,Category,Sales
0,1,CA-2016-152156,11/8/2016,Claire Gute,South,Furniture,261.9600
1,2,CA-2016-152156,11/8/2016,Claire Gute,South,Furniture,731.9400
2,3,CA-2016-138688,6/12/2016,Darrin Van Huff,West,Office Supplies,14.6200
3,4,US-2015-108966,10/11/2015,Sean O'Donnell,South,Furniture,957.5775
4,5,US-2015-108966,10/11/2015,Sean O'Donnell,South,Office Supplies,22.3680


> **Insight:** The schema successfully loaded with transactional granularity. Each row represents a specific line item in an order, capturing geographical data, customer details, categorical hierarchies, and financial metrics (Sales, Profit, Discount).

## Step 3: Targeted Filtering (WHERE Clauses)
**Objective:** Isolate high-value segments, specifically analyzing the "Technology" category within the "West" region to understand targeted sales performance.

In [3]:
# Example: Filter for Technology sales in the West region
query_filter = """
SELECT Order_ID, Order_Date, Customer_Name, Category, Sales, Region
FROM sales
WHERE Region = 'West' AND Category = 'Technology'
LIMIT 10;
"""
display(pd.read_sql_query(query_filter, conn))

,Order_ID,Order_Date,Customer_Name,Category,Sales,Region
0,CA-2014-115812,6/9/2014,Brosina Hoffman,Technology,907.152,West
1,CA-2014-115812,6/9/2014,Brosina Hoffman,Technology,911.424,West
2,CA-2014-143336,8/27/2014,Zuschuss Donatelli,Technology,213.480,West
3,CA-2016-121755,1/16/2016,Eric Hoffmann,Technology,90.570,West
4,CA-2015-135545,11/24/2015,Kunst Miller,Technology,13.980,West
5,CA-2014-106376,12/5/2014,Brendan Sweed,Technology,167.968,West
6,CA-2016-109806,9/17/2016,Jim Sink,Technology,73.584,West
7,US-2015-156867,11/13/2015,Lena Cacioppo,Technology,238.896,West
8,CA-2016-158834,3/13/2016,Tamara Willingham,Technology,203.184,West
9,CA-2014-123260,8/26/2014,Frank Merwin,Technology,176.800,West


> **Insight:** Filtering for the Technology sector in the West region reveals transactions that generally command higher individual sales values compared to standard office supplies, indicating this segment is crucial for regional revenue generation.

## Step 4: Categorical Aggregations
**Objective:** Calculate overarching metrics—Total Sales, Total Quantity, and Average Discount—grouped by main product categories to identify volume vs. revenue drivers.

In [4]:
# Calculate Total Sales, Total Quantity, and Average Discount by Category
query_agg = """
SELECT Category, 
       ROUND(SUM(Sales), 2) as Total_Sales, 
       SUM(Quantity) as Total_Quantity, 
       ROUND(AVG(Discount), 4) as Avg_Discount
FROM sales
GROUP BY Category;
"""
display(pd.read_sql_query(query_agg, conn))

,Category,Total_Sales,Total_Quantity,Avg_Discount
0,Furniture,741999.80,8028,0.1739
1,Office Supplies,719047.03,22906,0.1573
2,Technology,836154.03,6939,0.1323


> **Insight:** There is a distinct split between volume and revenue. While 'Office Supplies' drives the highest transaction volume (Quantity), the 'Technology' and 'Furniture' categories are the primary drivers of total revenue, despite lower unit sales.

## Step 5: Top-Performing Products
**Objective:** Rank the Sub-Categories by Total Sales to pinpoint the most lucrative product lines.

In [5]:
# Top 5 Sub-Categories by Sales
query_sort = """
SELECT Sub_Category, ROUND(SUM(Sales), 2) as Total_Sales
FROM sales
GROUP BY Sub_Category
ORDER BY Total_Sales DESC
LIMIT 5;
"""
display(pd.read_sql_query(query_sort, conn))

,Sub_Category,Total_Sales
0,Phones,330007.05
1,Chairs,328449.10
2,Storage,223843.61
3,Tables,206965.53
4,Binders,203412.73


> **Insight:** 'Phones' and 'Chairs' dominate the top spots. This suggests that enterprise-level hardware and office furniture are the core pillars of the Superstore's gross sales.

## Step 6: Advanced Business Use Cases - Customer Profitability
**Objective:** Identify the top 10 most profitable customers to inform potential loyalty programs or VIP account management strategies.

In [6]:
# Top 10 Customers by Total Profit
query_customers = """
SELECT Customer_Name, ROUND(SUM(Profit), 2) as Total_Profit
FROM sales
GROUP BY Customer_Name
ORDER BY Total_Profit DESC
LIMIT 10;
"""
display(pd.read_sql_query(query_customers, conn))

,Customer_Name,Total_Profit
0,Tamara Chand,8981.32
1,Raymond Buch,6976.10
2,Sanjit Chand,5757.41
3,Hunter Lopez,5622.43
4,Adrian Barton,5444.81
5,Tom Ashbrook,4703.79
6,Christopher Martinez,3899.89
7,Keith Dawkins,3038.63
8,Andy Reiter,2884.62
9,Daniel Raglin,2869.08


> **Insight:** A significant portion of overall profit is concentrated among a small group of VIP clients. Ensuring high retention rates for these specific individuals should be a strategic priority.

## Step 7: Data Quality and Validation
**Objective:** Verify data integrity by confirming the final database row count matches the source file expectations.

In [7]:
query_validate = """
SELECT COUNT(*) as Total_Rows 
FROM sales;
"""
display(pd.read_sql_query(query_validate, conn))

,Total_Rows
0,9994


> **Insight:** The SQL row count returns 9,994 rows, perfectly matching the original CSV dataset. This validates that no data was dropped or duplicated during the Pandas-to-SQLite ingestion process.